# Flood Prediction Model

This model uses satellite imagery as RGB inputs to a machine learning tool to determine areas that are flooded vs not flooded. The example image is of Brzeg, Poland, on 18 September 2024.

The imagery is from the Sentinel-2 satellite (downloaded from the Copernicus Browser). QGIS was used for visualisation and for creating the ground-truth points.

**The work is divided into three execution blocks:**
1. **Execution 1:** the ground-truth points (GTP) are used for both training and testing 4 different machine learning models. The most successful model is used in Execution 2.
2. **Execution 2:** all GTP are used to train the chosen model. The trained model is then used on new points with unknown flooding classifications, and the result is mapped.
3. **Execution 3:** evaluation of the learned model against ground-truth polygons (not yet completed).

**How to use this notebook**
1. Put your input files in a data folder (see the Settings cell for the names).
2. Edit the **Settings** cell only. Everything else should run as it is.
3. Run the notebook from top to bottom.

**Input files needed**
- A Sentinel-2 RGB image of the area of interest (`.tif`, bands 1-3 = R, G, B)
- A GeoPackage of labelled ground-truth points (one column holding 1 = flooded, 0 = not flooded)
- A GeoPackage of unlabelled points (one per pixel centroid) to classify

## Install & import necessary packages

In [ ]:
!pip install -q rasterio geopandas scikit-learn matplotlib

In [ ]:
from pathlib import Path

import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

import rasterio
from rasterio.enums import Resampling
from rasterio.plot import plotting_extent

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC

## Settings
**Edit this cell only.** All file paths and options are set here.

In [ ]:
# (Optional) On Google Colab, uncomment these two lines to access files in Google Drive:
# from google.colab import drive
# drive.mount("/content/drive")

# Folder containing the input files (on Colab, e.g. Path("/content/drive/MyDrive/Final_Assignment"))
DATA_DIR = Path("data")

# Input files
INPUT_TIF = DATA_DIR / "brzeg_clip.tif"               # satellite image of the area of interest
GROUND_TRUTH_POINTS = DATA_DIR / "y_variable2.gpkg"   # labelled points used for training/testing
PIXEL_POINTS = DATA_DIR / "Centroids2.gpkg"           # unlabelled pixel centroids to classify

# Options
CLASS_HEADING = "flooded"       # attribute column holding the label (1 = flooded, 0 = not flooded)
RESAMPLING_FACTOR = 7           # the image is shrunk by this factor (1 = no resampling)
BEST_MODEL = "Random Forest"    # model used in Execution 2, chosen after Execution 1
TOWN, COUNTRY = "Brzeg", "Poland"   # used in the map title (TOWN can be None)
RANDOM_STATE = 42               # makes results repeatable

# Output files (created automatically)
OUTPUT_DIR = DATA_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESAMPLED_TIF = OUTPUT_DIR / f"{INPUT_TIF.stem}_resampled{RESAMPLING_FACTOR}.tif"
PREDICTED_POINTS = OUTPUT_DIR / "predicted_points.gpkg"
FLOODED_POINTS = OUTPUT_DIR / "flooded_points.gpkg"
PREDICTED_POLYGONS = OUTPUT_DIR / "predicted_polygons.gpkg"

## Load and prepare data

In [ ]:
def tif_import_and_resample(input_file_path, resampling_factor, output_file_path,
                            method=Resampling.cubic):
    """Resample a tif image to a lower resolution and save it.

    Inputs:
    1. input_file_path: file path of the area of interest
    2. resampling_factor: factor by which the image is shrunk (e.g. 7 = 7x fewer pixels per side)
    3. output_file_path: file path of where to save the output
    4. method: resampling method. Cubic suits continuous RGB values. Use
       Resampling.nearest to keep the original pixel values unchanged.
    """
    with rasterio.open(input_file_path) as src:
        profile = src.profile
        new_width = src.width // resampling_factor
        new_height = src.height // resampling_factor
        resampled_data = src.read(out_shape=(src.count, new_height, new_width),
                                  resampling=method)
        new_transform = src.transform * src.transform.scale(src.width / new_width,
                                                            src.height / new_height)
        profile.update({"width": new_width, "height": new_height, "transform": new_transform})

    with rasterio.open(output_file_path, "w", **profile) as dst:
        dst.write(resampled_data)


def extract_CRS(tif_file_path):
    """Return the coordinate reference system (CRS) of a tif image.

    Input: tif_file_path: file path of the (resampled or not) area of interest.
    Returns the EPSG code, or the CRS as text if it has no EPSG code.
    """
    with rasterio.open(tif_file_path) as src:
        return src.crs.to_epsg() or src.crs.to_wkt()


def get_pixel_size(tif_file_path):
    """Return the pixel width of a tif image, in the units of its CRS (usually metres)."""
    with rasterio.open(tif_file_path) as src:
        return abs(src.res[0])


def extract_points(points_file, tif_file):
    """Extract the RGB values at each point of a GeoPackage.

    Note that for labelled points, flooded is indicated by a 1 and not-flooded by a 0.

    Inputs:
    1. points_file: file path of the points GeoPackage
    2. tif_file: file path of the (resampled or not) tif image

    Returns:
    x_var: array of RGB values, one row per point
    y_var: GeoDataFrame of the points (keeps geometry and attributes)
    """
    y_var = gpd.read_file(points_file)
    with rasterio.open(tif_file) as src:
        # Make sure the points and the image use the same coordinate system
        if y_var.crs is not None and src.crs is not None and y_var.crs != src.crs:
            y_var = y_var.to_crs(src.crs)
        coords = list(zip(y_var.geometry.x, y_var.geometry.y))
        x_var = np.array(list(src.sample(coords, indexes=[1, 2, 3])), dtype=np.uint8)
    return x_var, y_var

In [ ]:
# Resample the image once (delete the output file to redo it)
if not RESAMPLED_TIF.exists():
    tif_import_and_resample(INPUT_TIF, RESAMPLING_FACTOR, RESAMPLED_TIF)

## Part 1.1: Training

The flood prediction model is tested and trained with 4 different classifiers:
1. Random Forest
2. Logistic Regression
3. Support Vector Classifier (SVC)
4. Multi-Layer Perceptron (MLP)

**Variables**
- `y_variable` holds the classification of each point: flooded = 1, not flooded = 0
- `x_variable` holds the RGB values at each of those points

Each function below needs 2 inputs: `train_X` (RGB values of the training points) and `train_y` (their flooded / not-flooded labels).

In [ ]:
def learn_rand_forest(train_X, train_y):
    model = RandomForestClassifier(random_state=RANDOM_STATE)
    model.fit(train_X, train_y)
    return model


def learn_logistic_regression(train_X, train_y):
    model = LogisticRegression()
    model.fit(train_X, train_y)
    return model


def learn_SVC(train_X, train_y):
    model = SVC()
    model.fit(train_X, train_y)
    return model


def learn_MLP(train_X, train_y):
    # Note: the MLP can give inconsistent results between runs; random_state keeps it repeatable.
    model = MLPClassifier(max_iter=1000, random_state=RANDOM_STATE)
    model.fit(train_X, train_y)
    return model


# Model names that can be used in BEST_MODEL, and the function that trains each one
MODEL_LEARNERS = {
    "Random Forest": learn_rand_forest,
    "Logistic Regression": learn_logistic_regression,
    "SVC": learn_SVC,
    "MLP": learn_MLP,
}

## Part 1.2: Evaluation

To decide which model is best at detecting flooded areas from RGB values, each model is scored on the testing data with two performance indicators:
1. F1 score
2. Confusion matrix

Both functions need: a trained `model`, the testing points `test_X` and `test_y`, and the `model_name` (used in the printout and plot title).

In [ ]:
def get_f1_score(model, test_X, test_y, model_name):
    """Print the F1 score of a trained model for each class."""
    y_prediction = model.predict(test_X)
    score = f1_score(test_y, y_prediction, average=None)
    print("\nThe F1 score for the {} method in predicting:\n"
          "the not flooded areas: {:.3f}\n"
          "the flooded areas: {:.3f}".format(model_name, score[0], score[1]))
    return score


def get_confusion_matrix(model, test_X, test_y, model_name):
    """Plot the confusion matrix of a trained model."""
    y_prediction = model.predict(test_X)
    ConfusionMatrixDisplay.from_predictions(
        test_y, y_prediction, labels=[0, 1],
        display_labels=["Not flooded", "Flooded"], colorbar=False, cmap="RdYlGn")
    plt.title("Confusion Matrix for {}".format(model_name))
    plt.show()

## Execution 1

The ground-truth points are split into training and testing sets and used to train and test each model.
The best-performing model is then chosen for Execution 2 (set `BEST_MODEL` in the Settings cell).

- 80% of the points are used for training (the models learn from these)
- 20% of the points are held back for testing

In [ ]:
x_variable, y_variable_geo = extract_points(GROUND_TRUTH_POINTS, RESAMPLED_TIF)
y_variable = y_variable_geo[CLASS_HEADING]
# Note: y_variable_geo keeps the geometry, whereas y_variable is used for calculation

X_train, X_test, y_train, y_test = train_test_split(
    x_variable, y_variable, test_size=0.2, stratify=y_variable, random_state=RANDOM_STATE)

# Train and test each model, then print its F1 score and confusion matrix
for model_name, learner in MODEL_LEARNERS.items():
    model = learner(X_train, y_train)
    get_f1_score(model, X_test, y_test, model_name)
    get_confusion_matrix(model, X_test, y_test, model_name)

## Part 2.1: Application

The aim of this section is to classify unknown points as flooded or not, using the chosen model (`BEST_MODEL`).

In [ ]:
def prediction(model_name, pixel_x, pixel_geo, training_X, training_y, epsg, output_path,
               class_heading="flooded"):
    """Train a model on all ground-truth points and classify the unlabelled pixel points.

    Inputs:
    1. model_name: one of the names in MODEL_LEARNERS
    2. pixel_x, pixel_geo: RGB values and GeoDataFrame of the points to classify
    3. training_X, training_y: RGB values and labels of the ground-truth points
    4. epsg: coordinate reference system to save the predictions in
    5. output_path: GeoPackage file path to save the predictions
    6. class_heading: name of the column that will hold the predicted label
    """
    model = MODEL_LEARNERS[model_name](training_X, training_y)
    predicted = pixel_geo.copy()
    predicted[class_heading] = model.predict(pixel_x)
    predicted = predicted.set_crs(epsg, allow_override=True)
    predicted.to_file(output_path, layer="predicted_points", driver="GPKG")
    print("Predicted {} flooded and {} not flooded points.".format(
        int((predicted[class_heading] == 1).sum()), int((predicted[class_heading] == 0).sum())))


def predicted_to_flooded(predictions, output_path, heading):
    """Keep only the points predicted as flooded and save them.

    Inputs:
    1. predictions: file path of the predicted points GeoPackage
    2. output_path: file path to save the flooded points
    3. heading: name of the column holding the flooded / not-flooded label
    """
    gdf = gpd.read_file(predictions)
    filtered_points = gdf[gdf[heading] == 1]
    if filtered_points.empty:
        raise ValueError("No points were predicted as flooded.")
    filtered_points.to_file(output_path, driver="GPKG")


def points_to_polygons(points_file_path, output_path, pixel_size):
    """Turn each point into a square polygon the size of one pixel.

    The points are the centres of the resampled image pixels. To represent the whole
    pixel, not just its centre, each point is buffered by half the pixel width, so the
    square has the same width as a pixel.

    Inputs:
    1. points_file_path: file path of the flooded points GeoPackage
    2. output_path: file path to save the polygons
    3. pixel_size: pixel width in CRS units (use get_pixel_size())
    """
    gdf = gpd.read_file(points_file_path)
    gdf["geometry"] = gdf.geometry.buffer(pixel_size / 2, cap_style="square")
    gdf.to_file(output_path, layer="flooded_polygons", driver="GPKG")


def map_viewer(input_path, image_path, type_, town, country):
    """Plot the flooded areas over the satellite image.

    Inputs:
    1. input_path: file path of the GeoPackage with only flooded points or polygons
    2. image_path: file path of the (resampled) satellite image
    3. type_: "points" or "polygons"
    4. town: the township shown on the map (or None)
    5. country: the country shown on the map
    """
    with rasterio.open(image_path) as src:
        raster_data = src.read([1, 2, 3])
        crs = src.crs
        extent = plotting_extent(src)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(raster_data.transpose(1, 2, 0), extent=extent)

    flooded = gpd.read_file(input_path).to_crs(crs)
    if type_ == "points":
        flooded.plot(ax=ax, color="blue", markersize=2)
    elif type_ == "polygons":
        flooded.plot(ax=ax, facecolor="blue", alpha=0.9, edgecolor="blue")
    else:
        raise ValueError('type_ must be "points" or "polygons"')
    ax.legend(handles=[Patch(facecolor="blue", edgecolor="blue", label="Flooded area")])

    place = country if town is None else "{}, {}".format(town, country)
    plt.title("Sentinel-2 Satellite Image of {}".format(place), fontsize=11)
    plt.figtext(0.1, 0.08, "{}".format(crs), ha="left", fontsize=8)
    ax.set_xlabel("Easting" if crs.is_projected else "Longitude")
    ax.set_ylabel("Northing" if crs.is_projected else "Latitude")
    plt.show()

## Execution 2

1. Extract the RGB values of the ground-truth points (GTP) and of the new pixel points to classify.
2. Train the chosen model on **all** GTP and predict the flooding of the pixel points. The results are saved as vector points, one per pixel of the satellite image.
3. Keep the coordinate system of the tif image so the output is consistent with it.
4. Extract the flooded points, turn them into pixel-sized polygons, and view them over the satellite image.

In [ ]:
# "GTP_" prefix = training points. All of them are used for training here.
GTP_x, GTP_y_geo = extract_points(GROUND_TRUTH_POINTS, RESAMPLED_TIF)
GTP_y = GTP_y_geo[CLASS_HEADING]

# New points (pixels) with unknown flooding classification
pixel_x, pixel_geo = extract_points(PIXEL_POINTS, RESAMPLED_TIF)

# Keep the coordinate system of the image
EPSG = extract_CRS(RESAMPLED_TIF)

# Predict, then keep only the flooded points
prediction(BEST_MODEL, pixel_x, pixel_geo, GTP_x, GTP_y, EPSG, PREDICTED_POINTS, CLASS_HEADING)
predicted_to_flooded(PREDICTED_POINTS, FLOODED_POINTS, CLASS_HEADING)

# Turn the flooded points into polygons and map them
points_to_polygons(FLOODED_POINTS, PREDICTED_POLYGONS, get_pixel_size(RESAMPLED_TIF))
map_viewer(PREDICTED_POLYGONS, RESAMPLED_TIF, "polygons", TOWN, COUNTRY)

## Execution 3: Evaluation (to be completed)

Evaluation of the learned model against independent ground-truth polygons.

Note: the predictions in Execution 2 cannot be scored against the training points, because those points were used to train the model. The F1 scores from Execution 1 (held-out test points) are the current estimate of performance.